# Kern QLoRA - Colab T4 (Recovery-Safe)

Run this notebook in order.
It auto-recovers broken working directory state and re-clones the repo.


In [ ]:
# 1) Recover cwd + ensure fresh repo in /content/kern
import os
os.chdir('/')
os.makedirs('/content', exist_ok=True)
%cd /content

if os.path.isdir('/content/kern/.git'):
    !git -C /content/kern fetch --all --prune
    !git -C /content/kern reset --hard origin/main
else:
    !rm -rf /content/kern
    !git clone https://github.com/OscarCode9/kern.git /content/kern

%cd /content/kern
!pwd
!ls -la train_qwen_qlora_t4.py notebooks/train_kern_colab.ipynb


In [ ]:
# 2) Verify GPU
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not detected. In Colab: Runtime -> Change runtime type -> T4 GPU, then restart runtime.')


In [ ]:
# 3) Install dependencies
!python -m pip install -U pip
!pip install torch transformers datasets peft trl accelerate bitsandbytes sentencepiece


In [ ]:
# 4) Build dataset only if missing
import os
if not os.path.exists('data/finetune_csn20k_v2/train_qwen_chat.jsonl'):
    !python prepare_finetune_dataset_csn.py --target-kept 20000 --valid-ratio 0.05 --out-dir data/finetune_csn20k_v2 --overwrite
else:
    print('Dataset already exists: data/finetune_csn20k_v2')


In [ ]:
# 5) T4-safe training run (resume supported)
!python train_qwen_qlora_t4.py \
  --model-name Qwen/Qwen3.5-9B \
  --train-file data/finetune_csn20k_v2/train_qwen_chat.jsonl \
  --valid-file data/finetune_csn20k_v2/valid_qwen_chat.jsonl \
  --output-dir outputs/qwen-kern-qlora-t4 \
  --max-train-samples 5000 \
  --max-valid-samples 500 \
  --num-train-epochs 1 \
  --per-device-batch-size 1 \
  --gradient-accumulation-steps 16 \
  --max-seq-length 512 \
  --save-steps 100 \
  --eval-steps 100 \
  --resume-from-checkpoint auto
